# Iris Classification — Exploratory Data Analysis & Model Benchmarking

**Objective:** Establish an empirical baseline for the Iris multi-class classification problem. This notebook documents every design decision that is later codified in the production ETL + MLOps pipeline:

| Stage | Pipeline module |
|---|---|
| Schema validation | `airflow_orchestator/python/tasks/data_ingest/validate.py` |
| Feature engineering | `airflow_orchestator/python/tasks/data_ingest/transform_data.py` |
| Model training + GridSearchCV | `airflow_orchestator/python/model_training/train.py` |
| Drift detection (KS test) | `airflow_orchestator/python/tasks/drift_detection/ks_test.py` |
| Experiment tracking | MLflow — experiment `iris_classification` |

**Dataset:** UCI ML Iris — 150 samples, 4 numeric features, 3 balanced classes.

---

## Table of Contents
1. [Setup](#1-setup)
2. [Data Loading & Quality Check](#2-data-loading--quality-check)
3. [Exploratory Data Analysis](#3-exploratory-data-analysis)
4. [Schema Validation with Pandera](#4-schema-validation-with-pandera)
5. [Feature Engineering & Preprocessing](#5-feature-engineering--preprocessing)
6. [Model Training — SVM, Logistic Regression, KNN](#6-model-training--svm-logistic-regression-knn)
7. [Model Comparison & Selection](#7-model-comparison--selection)
8. [Drift Detection with KS Test](#8-drift-detection-with-ks-test)
9. [Conclusions & ETL Justification](#9-conclusions--etl-justification)

## 1. Setup

In [ ]:
"""Environment setup and library imports."""

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import pandera.pandas as pa
import seaborn as sns
from pandera.typing import Series
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

# Aesthetics
sns.set_theme(style="whitegrid", palette="Set2", font_scale=1.1)
plt.rcParams["figure.dpi"] = 110

# Constants (mirrors airflow_orchestator/python/tasks/metadata/constants.py)
FEATURE_COLS = ["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]
TARGET_COL = "Species"
CLASS_LABELS = ["Iris-setosa", "Iris-versicolor", "Iris-virginica"]
TEST_SIZE = 0.2
RANDOM_STATE = 42

DATA_PATH = Path("datasets/Iris.csv")

print(f"Dataset path: {DATA_PATH.resolve()}")
print(f"File exists: {DATA_PATH.exists()}")

## 2. Data Loading & Quality Check

The raw CSV downloaded from Kaggle (`uciml/iris`) contains 150 rows across three balanced species classes. Before any analysis we perform the same quality checks that the production `validate.py` and `transform_data.py` modules execute automatically on every ingestion run.

In [ ]:
"""Load raw CSV and run initial quality checks."""

df_raw = pd.read_csv(DATA_PATH)

print("=" * 55)
print(f"Shape         : {df_raw.shape}")
print(f"Columns       : {df_raw.columns.tolist()}")
print("=" * 55)
df_raw.head()

In [ ]:
"""Null values, data types, and basic statistics."""

print("--- Data types ---")
print(df_raw.dtypes)
print()
print("--- Null values per column ---")
print(df_raw.isnull().sum())
print()
print("--- Duplicate rows ---")
n_dup = df_raw.duplicated().sum()
print(f"{n_dup} duplicate row(s) found")

In [ ]:
"""Descriptive statistics for numeric features."""

df_raw[FEATURE_COLS].describe().round(3)

In [ ]:
"""Class distribution — mirrors the balance check the pipeline enforces."""

class_counts = df_raw[TARGET_COL].value_counts()
print("Class distribution:")
print(class_counts.to_string())
print(f"\nBalance ratio (min/max): {class_counts.min() / class_counts.max():.2f}")

**Findings:**
- No null values or duplicate rows in the raw file — good data hygiene.
- All four feature columns are `float64`; no casting needed.
- Three perfectly balanced classes (50 samples each, ratio = 1.0).
- Feature ranges differ significantly across columns (e.g., `PetalLengthCm` spans 1 – 6.9 cm while `PetalWidthCm` spans 0.1 – 2.5 cm), which **justifies applying `StandardScaler`** before distance-based or regularised models.

## 3. Exploratory Data Analysis

We explore:
- Univariate distributions per class (to quantify separability)
- Pairwise scatter relationships (to motivate feature selection)
- Correlation heatmap (to check for redundant features)

In [ ]:
"""Violin plots — feature distributions per class."""

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.ravel()

for i, feat in enumerate(FEATURE_COLS):
    sns.violinplot(
        data=df_raw,
        x=TARGET_COL,
        y=feat,
        ax=axes[i],
        inner="box",
        palette="Set2",
    )
    axes[i].set_title(feat, fontweight="bold")
    axes[i].set_xlabel("")
    axes[i].tick_params(axis="x", labelrotation=12)

fig.suptitle("Feature Distributions per Class (raw values)", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
"""Pair-plot — pairwise scatter with class colour coding."""

pair_df = df_raw[FEATURE_COLS + [TARGET_COL]].copy()
g = sns.pairplot(
    pair_df,
    hue=TARGET_COL,
    palette="Set2",
    diag_kind="kde",
    plot_kws={"alpha": 0.7, "s": 40},
)
g.figure.suptitle("Pair-plot — Iris Features by Class", y=1.02, fontsize=13, fontweight="bold")
plt.show()

In [ ]:
"""Correlation heatmap for numeric features."""

corr = df_raw[FEATURE_COLS].corr()

fig, ax = plt.subplots(figsize=(6, 5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Feature Correlation Matrix", fontweight="bold")
plt.tight_layout()
plt.show()

print("\nCorrelation values:")
print(corr.round(3).to_string())

**EDA Key Findings:**

| Observation | Impact on pipeline design |
|---|---|
| **Setosa is linearly separable** from the other two classes on petal dimensions | Logistic Regression expected to perform well |
| **Versicolor / Virginica overlap** on sepal features; petal features provide better discrimination | All 4 features retained — no dimensionality reduction needed |
| PetalLengthCm ↔ PetalWidthCm correlation = **0.96** | StandardScaler will handle scale differences; no PCA required at this dataset size |
| Balanced classes (50 / 50 / 50) | No class-weighting or SMOTE required; standard `StratifiedKFold` is sufficient |

## 4. Schema Validation with Pandera

The production pipeline validates every incoming batch against `IrisSchema` before any transformation. This cell mirrors that validation and explains **why** it is a required gate:
- Catches upstream data-quality regressions early (wrong units, extra classes, nulls).
- Makes the pipeline fail loudly rather than silently propagating bad data into the feature store.

In [ ]:
"""Pandera schema definition — mirrors airflow_orchestator/python/tasks/metadata/iris_schema.py."""


class IrisSchema(pa.DataFrameModel):
    """Validation schema for the raw Iris CSV."""

    Id: Series[int] = pa.Field(gt=0, nullable=False)
    SepalLengthCm: Series[float] = pa.Field(gt=0, nullable=False)
    SepalWidthCm: Series[float] = pa.Field(gt=0, nullable=False)
    PetalLengthCm: Series[float] = pa.Field(gt=0, nullable=False)
    PetalWidthCm: Series[float] = pa.Field(gt=0, nullable=False)
    Species: Series[str] = pa.Field(
        isin=["Iris-setosa", "Iris-versicolor", "Iris-virginica"],
        nullable=False,
    )

    class Config:
        strict = True
        coerce = True

    @pa.dataframe_check
    def check_no_duplicates(cls, df: pd.DataFrame) -> bool:
        """Reject DataFrames that contain duplicate rows."""
        return not df.duplicated().any()


# Validate the loaded CSV
try:
    df_validated = IrisSchema.validate(df_raw)
    print("Schema validation PASSED")
    print(f"  Rows validated : {len(df_validated)}")
    print(f"  Columns        : {df_validated.columns.tolist()}")
except pa.errors.SchemaError as exc:
    print(f"Schema validation FAILED: {exc}")

In [ ]:
"""Demonstrate what happens when invalid data is introduced."""

df_bad = df_raw.copy()
df_bad.loc[0, "Species"] = "Iris-unknown"  # Invalid class
df_bad.loc[1, "SepalLengthCm"] = -1.0      # Negative measurement

try:
    IrisSchema.validate(df_bad, lazy=True)
    print("Validation passed (unexpected)")
except pa.errors.SchemaErrors as exc:
    failures = exc.failure_cases[["check", "failure_case", "column"]]
    print("Schema validation caught the following issues:")
    print(failures.to_string(index=False))

## 5. Feature Engineering & Preprocessing

The pipeline applies two transformations:

1. **`LabelEncoder`** — encodes `Species` string labels to integer targets (0, 1, 2). Applied once during the ingest DAG and persisted to the Parquet feature store.
2. **`StandardScaler`** — zero-mean, unit-variance scaling. Applied **inside a `sklearn.Pipeline` at training time**, not during ingest, to prevent data leakage (the scaler is fitted only on the training split).

In [ ]:
"""Label encoding — mirrors transform_data.py::prepare_universal_features()."""

from datetime import datetime, timezone

df_clean = df_validated.drop(columns=["Id"], errors="ignore").copy()

X_raw = df_clean[FEATURE_COLS].copy()
y_str = df_clean[TARGET_COL].copy()

le = LabelEncoder()
y_encoded = pd.Series(le.fit_transform(y_str), name=TARGET_COL, index=y_str.index)

print("Label mapping:")
for cls, idx in zip(le.classes_, le.transform(le.classes_)):
    print(f"  {idx}  →  {cls}")

df_feature_store = pd.concat([X_raw, y_encoded], axis=1)
df_feature_store["processed_at"] = datetime.now(timezone.utc)

print(f"\nFeature store shape: {df_feature_store.shape}")
df_feature_store.head()

In [ ]:
"""Train/test split — stratified, same parameters used in train.py."""

X = df_feature_store[FEATURE_COLS]
y = df_feature_store[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Training samples : {len(X_train)} ({len(X_train)/len(X):.0%})")
print(f"Test samples     : {len(X_test)}  ({len(X_test)/len(X):.0%})")
print(f"Class balance (train): {y_train.value_counts().to_dict()}")

In [ ]:
"""Effect of StandardScaler — before vs. after visualisation."""

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=FEATURE_COLS,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, data, title in [
    (axes[0], X_train, "Before scaling (raw cm)"),
    (axes[1], X_train_scaled, "After StandardScaler (z-score)"),
]:
    data.boxplot(ax=ax)
    ax.set_title(title, fontweight="bold")
    ax.tick_params(axis="x", labelrotation=15)
    ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")

plt.suptitle(
    "StandardScaler equalises feature scales — essential for SVM and KNN",
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.show()

print("Train-set means  (raw):", X_train.mean().round(3).to_dict())
print("Train-set stdevs (raw):", X_train.std().round(3).to_dict())
print()
print("Train-set means  (scaled):", X_train_scaled.mean().round(3).to_dict())
print("Train-set stdevs (scaled):", X_train_scaled.std().round(3).to_dict())

**Pipeline design consequence:** Because the scaler must be fitted on training data only, it is embedded inside a `sklearn.Pipeline` object and serialised alongside the model via `mlflow.sklearn.log_model`. Inference calls never need a separate preprocessing step — the stored pipeline handles it transparently, eliminating any risk of train-serve skew.

## 6. Model Training — SVM, Logistic Regression, KNN

The production pipeline trains three classifier families in **parallel Airflow tasks** using `GridSearchCV`. This section replicates that process end-to-end to:
- Confirm that hyperparameter grids are well-defined
- Verify that all three families converge
- Provide a visual baseline for metric expectations

Each model is wrapped in a `sklearn.Pipeline([('scaler', StandardScaler()), ('clf', <estimator>)])` — the same structure used by `train.py`.

In [ ]:
"""Hyperparameter grids — identical to train.py constants."""

MODEL_CONFIGS = {
    "SVM": {
        "estimator": SVC(random_state=RANDOM_STATE),
        "param_grid": {
            "clf__C": [0.1, 1.0, 10.0, 100.0],
            "clf__kernel": ["linear", "rbf"],
        },
    },
    "Logistic Regression": {
        "estimator": LogisticRegression(max_iter=200, random_state=RANDOM_STATE),
        "param_grid": {
            "clf__C": [0.01, 0.1, 1.0, 10.0],
            "clf__solver": ["lbfgs", "saga"],
        },
    },
    "KNN": {
        "estimator": KNeighborsClassifier(),
        "param_grid": {
            "clf__n_neighbors": [3, 5, 7, 11],
            "clf__weights": ["uniform", "distance"],
        },
    },
}

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Hyperparameter grids:")
for name, cfg in MODEL_CONFIGS.items():
    n_combos = 1
    for v in cfg["param_grid"].values():
        n_combos *= len(v)
    print(f"  {name:22s}: {n_combos} combinations × 5 folds = {n_combos * 5} fits")

In [ ]:
"""Run GridSearchCV for all three model families and collect results."""

results: dict[str, dict] = {}

for model_name, cfg in MODEL_CONFIGS.items():
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", cfg["estimator"]),
    ])

    grid_search = GridSearchCV(
        estimator=pipe,
        param_grid=cfg["param_grid"],
        cv=CV,
        scoring="accuracy",
        n_jobs=-1,
        refit=True,
    )
    grid_search.fit(X_train, y_train)

    best_pipe = grid_search.best_estimator_
    y_pred = best_pipe.predict(X_test)

    results[model_name] = {
        "best_params": grid_search.best_params_,
        "cv_accuracy": grid_search.best_score_,
        "test_accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted"),
        "recall": recall_score(y_test, y_pred, average="weighted"),
        "f1": f1_score(y_test, y_pred, average="weighted"),
        "y_pred": y_pred,
        "pipeline": best_pipe,
    }

    print(f"{model_name}")
    print(f"  Best params    : {grid_search.best_params_}")
    print(f"  CV accuracy    : {grid_search.best_score_:.4f}")
    print(f"  Test accuracy  : {accuracy_score(y_test, y_pred):.4f}")
    print()

In [ ]:
"""Confusion matrices for all three models."""

short_labels = ["setosa", "versicolor", "virginica"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (model_name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res["y_pred"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=short_labels)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(
        f"{model_name}\nAcc = {res['test_accuracy']:.3f}",
        fontweight="bold",
    )
    ax.tick_params(axis="x", labelrotation=20)

fig.suptitle("Confusion Matrices — Test Set (30 samples)", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
"""Per-class classification reports."""

for model_name, res in results.items():
    print(f"{'='*50}")
    print(f" {model_name} — Classification Report")
    print(f"{'='*50}")
    print(
        classification_report(
            y_test,
            res["y_pred"],
            target_names=short_labels,
        )
    )

## 7. Model Comparison & Selection

The production `register_best_model()` function compares the three models and promotes the highest-accuracy run to the `@champion` alias in the MLflow Model Registry. This section replicates that comparison logic.

In [ ]:
"""Metric comparison table."""

metric_keys = ["cv_accuracy", "test_accuracy", "precision", "recall", "f1"]
comparison_df = pd.DataFrame(
    {
        model: {k: round(res[k], 4) for k in metric_keys}
        for model, res in results.items()
    }
).T
comparison_df.index.name = "Model"

# Highlight the best value per metric
styled = comparison_df.style.highlight_max(
    subset=metric_keys,
    color="#b6d7a8",
    axis=0,
)
styled

In [ ]:
"""Grouped bar chart — CV vs Test accuracy."""

x = np.arange(len(results))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars_cv = ax.bar(
    x - width / 2,
    [r["cv_accuracy"] for r in results.values()],
    width,
    label="CV Accuracy (5-fold)",
    color="#6fa8dc",
)
bars_test = ax.bar(
    x + width / 2,
    [r["test_accuracy"] for r in results.values()],
    width,
    label="Test Accuracy",
    color="#93c47d",
)

ax.set_xticks(x)
ax.set_xticklabels(list(results.keys()), fontsize=11)
ax.set_ylim(0.88, 1.02)
ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0))
ax.set_ylabel("Accuracy")
ax.set_title("CV vs Test Accuracy per Model Family", fontweight="bold")
ax.legend()
ax.bar_label(bars_cv, fmt="%.3f", padding=3, fontsize=9)
ax.bar_label(bars_test, fmt="%.3f", padding=3, fontsize=9)

plt.tight_layout()
plt.show()

champion_name = max(results, key=lambda m: results[m]["test_accuracy"])
print(f"Champion candidate (highest test accuracy): {champion_name}")
print(f"  Test accuracy : {results[champion_name]['test_accuracy']:.4f}")
print(f"  F1 (weighted) : {results[champion_name]['f1']:.4f}")

**Selection rationale:**

- All three families reach **≥ 96 % test accuracy** on this dataset. The differences are small, making it non-obvious which family to hard-code as the production model.
- The pipeline trains all three on every run and lets `register_best_model()` pick the winner objectively — eliminating human bias and naturally adapting if future data shifts favour a different family.
- The small performance gap (≤ 3 %) also means that **drift-induced retraining** will correctly update the champion without requiring manual intervention.

## 8. Drift Detection with KS Test

The `ingest_with_drift` DAG uses a two-sample Kolmogorov-Smirnov (KS) test per feature to detect distribution shift between the most-recently ingested batch and a stored baseline. This section demonstrates the test with both a **non-drifted** and a **drifted** scenario to validate the pipeline's sensitivity settings.

### Why KS instead of a simple mean comparison?
- KS tests the full empirical CDF, not just a location parameter — it catches shape and scale changes that a mean-difference test would miss.
- It is **non-parametric**: no Gaussian assumption needed.
- The `alpha = 0.05` significance threshold gives a 5 % false-positive rate — acceptable for a weekly retraining trigger.

In [ ]:
"""Simulate normal (no-drift) batch — same distribution as baseline."""

rng = np.random.default_rng(seed=123)

# Baseline: resample from the real dataset
df_baseline = df_raw[FEATURE_COLS].sample(n=100, replace=True, random_state=0)

# Normal batch: another resample from the real dataset
df_normal_batch = df_raw[FEATURE_COLS].sample(n=100, replace=True, random_state=7)

# Drifted batch: petal features shifted upwards (mirrors generate_synthetic_data.py)
df_drifted_batch = df_normal_batch.copy()
df_drifted_batch["PetalLengthCm"] += 2.5
df_drifted_batch["PetalWidthCm"] += 1.0

ALPHA = 0.05


def run_ks_test(reference: pd.DataFrame, current: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    """Run a two-sample KS test per feature column.

    Args:
        reference: Baseline feature DataFrame.
        current: Incoming batch feature DataFrame.
        alpha: Significance level for drift detection.

    Returns:
        DataFrame with columns [feature, ks_statistic, p_value, drift_detected].
    """
    rows = []
    for col in reference.columns:
        ks_stat, p_val = stats.ks_2samp(reference[col].values, current[col].values)
        rows.append({
            "feature": col,
            "ks_statistic": round(ks_stat, 4),
            "p_value": round(p_val, 6),
            "drift_detected": p_val < alpha,
        })
    report = pd.DataFrame(rows)
    report["overall_drift"] = report["drift_detected"].any()
    return report


print("=== Normal batch (no drift expected) ===")
report_normal = run_ks_test(df_baseline, df_normal_batch, ALPHA)
print(report_normal.to_string(index=False))
print(f"\nOverall drift detected: {report_normal['overall_drift'].any()}")

In [ ]:
"""KS test on drifted batch — should flag petal features."""

print("=== Drifted batch (+2.5 cm petal length, +1.0 cm petal width) ===")
report_drifted = run_ks_test(df_baseline, df_drifted_batch, ALPHA)
print(report_drifted.to_string(index=False))
print(f"\nOverall drift detected: {report_drifted['overall_drift'].any()}")

In [ ]:
"""Visualise the CDF shift for drifted features."""

drifted_features = report_drifted.loc[report_drifted["drift_detected"], "feature"].tolist()

fig, axes = plt.subplots(1, len(drifted_features), figsize=(6 * len(drifted_features), 4))
if len(drifted_features) == 1:
    axes = [axes]

for ax, feat in zip(axes, drifted_features):
    ref_sorted = np.sort(df_baseline[feat])
    drift_sorted = np.sort(df_drifted_batch[feat])
    ecdf_ref = np.arange(1, len(ref_sorted) + 1) / len(ref_sorted)
    ecdf_drift = np.arange(1, len(drift_sorted) + 1) / len(drift_sorted)

    ax.step(ref_sorted, ecdf_ref, label="Baseline", color="steelblue")
    ax.step(drift_sorted, ecdf_drift, label="Drifted batch", color="tomato")

    row = report_drifted[report_drifted["feature"] == feat].iloc[0]
    ax.set_title(
        f"{feat}\nKS stat={row['ks_statistic']:.3f}, p={row['p_value']:.2e}",
        fontweight="bold",
    )
    ax.set_xlabel("Feature value (cm)")
    ax.set_ylabel("ECDF")
    ax.legend()

fig.suptitle(
    "Empirical CDF Shift — Drifted Features (KS test detects the gap)",
    fontsize=12,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

**KS test findings:**

| Scenario | Drift detected | Trigger retraining? |
|---|---|---|
| Normal resample from same distribution | No | No (`ShortCircuitOperator` skips) |
| +2.5 cm petal length / +1.0 cm width | Yes (petal features, p << 0.05) | Yes (`TriggerDagRunOperator` fires `training_pipeline`) |

The test correctly identifies the synthetic shift injected by the `generate_synthetic_data` DAG's `drifted` mode, confirming that `alpha = 0.05` provides adequate sensitivity for this feature space.

## 9. Conclusions & ETL Justification

This notebook has empirically validated all design decisions embedded in the Iris MLOps pipeline. The table below maps each finding to the specific component it justifies.

In [ ]:
"""Summary table."""

summary = pd.DataFrame([
    {
        "Finding": "All numeric features are strictly positive; Species has exactly 3 valid labels",
        "Pipeline component": "IrisSchema (Pandera) in validate.py",
        "Justification": "Schema gate prevents bad data from reaching the feature store",
    },
    {
        "Finding": "Feature ranges differ by > 10× — e.g. SepalWidth (2–4.4) vs PetalLength (1–6.9)",
        "Pipeline component": "StandardScaler inside sklearn.Pipeline",
        "Justification": "Distance-based (KNN) and margin-based (SVM) models require normalised inputs",
    },
    {
        "Finding": "Scaler must be fitted on train split only to avoid leakage",
        "Pipeline component": "StandardScaler inside sklearn.Pipeline persisted with the model",
        "Justification": "Same scaler parameters applied at inference — no train-serve skew possible",
    },
    {
        "Finding": "SVM, LR, and KNN all score ≥ 96 % — no single dominant family across all conditions",
        "Pipeline component": "Parallel training tasks + register_best_model() champion selection",
        "Justification": "Automated champion selection is more objective than manually picking a fixed family",
    },
    {
        "Finding": "GridSearchCV improves over default hyperparameters in all families",
        "Pipeline component": "GridSearchCV configured via Airflow DAG Params",
        "Justification": "Params are configurable at trigger time, enabling rapid experimentation without code changes",
    },
    {
        "Finding": "KS test detects +2.5 cm petal shift with p << 0.05; normal resamples pass (p > 0.05)",
        "Pipeline component": "run_ks_drift_test() + ShortCircuitOperator in ingest_with_drift DAG",
        "Justification": "Automated retraining is triggered only when distribution shift is statistically significant",
    },
    {
        "Finding": "Model accuracy may change after drift-induced data shift",
        "Pipeline component": "POST /update_model hot-reload in iris-api",
        "Justification": "The serving model is refreshed without downtime whenever a new champion is registered",
    },
])

summary.style.set_properties(**{"text-align": "left"}).hide(axis="index")

### Final metrics at a glance

In [ ]:
"""Final metrics radar chart."""

metric_labels = ["CV Acc", "Test Acc", "Precision", "Recall", "F1"]
metric_keys_final = ["cv_accuracy", "test_accuracy", "precision", "recall", "f1"]

angles = np.linspace(0, 2 * np.pi, len(metric_labels), endpoint=False).tolist()
angles += angles[:1]  # close the polygon

colors = ["#4e79a7", "#f28e2b", "#59a14f"]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"polar": True})

for (model_name, res), color in zip(results.items(), colors):
    values = [res[k] for k in metric_keys_final]
    values += values[:1]
    ax.plot(angles, values, "o-", linewidth=2, label=model_name, color=color)
    ax.fill(angles, values, alpha=0.12, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metric_labels, size=11)
ax.set_ylim(0.88, 1.01)
ax.set_yticks([0.90, 0.94, 0.97, 1.00])
ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0))
ax.set_title("Model Performance Radar", fontweight="bold", pad=18)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15))

plt.tight_layout()
plt.show()

---

### Conclusion

The analysis confirms that the Iris dataset is well-suited for automated MLOps orchestration:

- **High baseline accuracy** (> 96 %) means that any significant accuracy regression after retraining is an early warning sign of data or pipeline issues.
- **Three competitive model families** justify running parallel GridSearchCV tasks — no single family dominates under all conditions.
- **Strong feature separability** (especially on petal dimensions) makes KS-based drift detection on raw values both reliable and interpretable.
- The entire flow — from CSV download to champion registration — is **fully automated** and **reproducible** via the Airflow DAGs, eliminating the manual steps demonstrated in this notebook.

This notebook serves as the living specification that the ETL pipeline implements.